# Part 5 — Inspect the existing job (ไม่เริ่มใหม่)

**รัน cell เดียวเพื่ออ่านผล job เดิม ไม่ใช่ rerun Part 5**

job ที่บันทึกไว้: `lab5-20261023-070538-degraded`
source commit, identities, roles และ compute ที่ทำไว้ยังคงอยู่

ตัวรอเดิมเทียบ `JobStatus.FAILED` กับ `Failed` ผิด ทำให้รอต่อหลัง job จบแล้ว ตอนนี้แก้ parser แล้ว
ยังไม่สรุปว่าตัว pipeline fail เพราะ gate ตามที่ตั้งใจ ต้องอ่าน child error/log ก่อน

cell นี้:
- อ่าน parent/child job statuses ครั้งเดียว
- ดาวน์โหลดเฉพาะ logs ของ failed child และพิมพ์ส่วนท้ายที่เกี่ยวข้อง
- ตรวจ model versions, endpoint และจำนวน compute nodes ปัจจุบัน
- เก็บ `reports/part5-inspection.json`

**ไม่มี submit/retrain, resource creation, role changes, Git commit, push หรือ cleanup**
ใช้ WSL/kernel `itcs355-lab3` เดิม ไม่ต้องรัน Parts 1–5 ใหม่


In [1]:
from pathlib import Path
import platform
import subprocess
import sys

if platform.system() != "Linux" or "microsoft" not in platform.release().lower():
    raise RuntimeError("เลือก kernel itcs355-lab3 ฝั่ง WSL ก่อนครับ")
here = Path.cwd().resolve()
lab = next((base / "lab5" for base in (here, *here.parents)
            if (base / "lab5/scripts/inspect_existing.py").is_file()
            and (base / "lab4/README.md").is_file()), None)
if lab is None:
    raise RuntimeError("เปิด notebook จาก repository เดิมก่อนครับ")
print("Inspect existing Part 5 job only — no setup, training or submission.", flush=True)
with subprocess.Popen(
    [sys.executable, "-u", "-m", "scripts.inspect_existing"], cwd=lab,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
) as task:
    try:
        for line in task.stdout:
            print(line, end="", flush=True)
        exit_code = task.wait()
    except KeyboardInterrupt:
        task.terminate()
        try:
            task.wait(timeout=10)
        except subprocess.TimeoutExpired:
            task.kill()
        raise
if exit_code:
    raise RuntimeError("อ่านผลเดิมไม่สำเร็จ ส่ง error ด้านบน ไม่ต้องรัน training ใหม่ครับ")
print("ส่ง SEND THIS SUMMARY กลับมา แล้วหยุดตรงนี้ครับ", flush=True)


Inspect existing Part 5 job only — no setup, training or submission.
[15:01:19] Read existing job and child statuses once: lab5-20261023-070538-degraded
[15:01:27] Read failed child: evaluate
[15:01:27] Download existing logs only (no models/data): 4548bb51-9347-4e11-a911-265f92605cb6
[15:01:35] Read registry versions, endpoint and current compute nodes; no changes

--- SEND THIS SUMMARY ---
{
  "inspected_at": "2026-10-23T08:01:27.289971+00:00",
  "job": "lab5-20261023-070538-degraded",
  "status": "Failed",
  "parent_error": null,
  "source_commit": "2487815e5bb65d7a216daf62366f254042174d32",
  "children": [
    {
      "name": "49e0088c-762e-4207-9924-0ee1574a8433",
      "step": "ingest",
      "status": "Completed"
    },
    {
      "name": "21af7d1c-bf4a-4b58-abd3-d40a27c06d03",
      "step": "validate",
      "status": "Completed"
    },
    {
      "name": "0819aee5-63bd-4fdb-aad0-cc440d5c5aed",
      "step": "train",
      "status": "Completed"
    },
    {
      "name": "454